**Dividing Study 1 participants by compulsivity**

We divide the 344 participants in the existing Study 1 selection into five quintiles based on Compulsivity: group 1 has the lowest scores and group 5 has the highest.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Update this folder when using the notebook on another computer.
project_dir = Path("/Users/daviddejonghe/Documents/Psychologie/Masterproef/simulationsHmfc")
symptom_file = project_dir / "Data" / "SymptomData.csv"
task_file = project_dir / "Data" / "PerceptualTaskData.csv"

# Save output in a separate folder under the current working directory.
output_dir = Path.cwd() / "study1_compulsivity_groepen"


In [2]:
keep_ids = [
     1109430,1136398,1161576,990377,988880,989613,989614,989638,990171,990502,
        1019534,1019545,1019829,1020003,1020094,1026298,1031230,1056155,1076128,
        1076202,1076245,1076467,1077091,1077672,1077748,1083215,1085767,1086215,
        1086429,1086524,1086722,1087676,1087803,1089279,1089330,1089829,1092255,
        1092315,1096800,1097476,1098724,1098967,1099520,1101067,1107685,1107856,
        1109466,1111115,1112029,1112249,1113314,1114743,1114859,944548,953958,
        954010,954564,957126,957158,958563,958619,959561,961534,965512,966909,
        967485,967866,969340,972471,974295,980324,984514,985216,988163,991011,
        998453,1029483,1029603,1029776,1029858,1030062,1030377,1030394,1030711,
        1031236,1032579,1060694,1071048,1072306,954995,955009,955021,955023,
        955077,955083,955102,955119,955240,955276,955303,955342,955383,955683,
        955984,956021,956352,956553,956843,956927,956993,957089,957167,957953,
        958100,958257,958360,958541,958577,958771,959130,959386,959708,960758,
        960817,961191,961560,961864,963283,964156,966004,966051,967144,967215,
        968578,968792,969242,969364,970091,970214,971440,971577,972235,972410,
        972412,972585,972608,972740,973276,973356,973496,973708,973896,974187,
        975356,977249,977374,978290,979523,979791,981984,982296,982316,982404,
        984377,990840,992115,992526,1000003,1002300,1003011,1005185,1055457,
        1055828,1056146,1056537,1057764,1057805,1058277,1058510,1058926,1059194,
        1059249,1059261,1059408,1059911,1060742,1063610,1067860,1069381,1069895,
        1069986,1071196,1071224,1071343,1072006,1072034,1072153,1072208,1074018,
        1074043,1074460,1076254,1077140,1077499,1077520,1077746,1079257,1080555,
        1080861,1081810,1083775,1084292,1084496,1085727,1086436,1087954,1088055,
        1089319,1089331,1089431,1089461,1094362,1094757,1099082,1099754,1100590,
        1102002,1105080,1105574,1106651,1107155,1113140,1113399,1117048,1123812,
        1139756,1164558,982442,982693,983044,983545,983866,983911,984204,984214,
        984218,984230,984337,984798,985205,985936,987475,1005286,1025813,1025854,
        1027427,1028174,1033318,1033745,1039326,1040390,1060306,1066153,1080350,
        1094893,970018,971334,971779,972231,972352,972382,972433,972550,973225,
        973843,974106,974217,974241,974337,974406,974604,974795,975381,975696,
        976140,976152,976614,976932,984212,985153,989292,992393,1000944,1003223,
        1003328,1008892,1008967,1009029,1052583,1059466,1060303,1060654,1061755,
        1066043,1066212,1066462,1066559,1066606,1066734,1068331,1068985,966047,
        969684,969744,969855,970770,973065,973458,1026348,1030251,1030362,1030403,
        1030520,1030648,1031514,1031608,1031873,1032457,1032513,1032530,1032639,
        1033030,1033395,1033397,1033887,1039513,1039632,1039920,1040014,1041483,
        1047104,1054199,1056861,1069540
]
assert len(keep_ids) == len(set(keep_ids)) == 344

In [3]:
sym = pd.read_csv(symptom_file)
task = pd.read_csv(task_file, usecols=["IDcode"])

for frame in (sym, task):
    frame["IDcode"] = pd.to_numeric(frame["IDcode"], errors="raise").astype("int64")

participants = sym.loc[sym["IDcode"].isin(keep_ids), ["IDcode", "Compulsivity"]].copy()
participants["Compulsivity"] = pd.to_numeric(participants["Compulsivity"], errors="raise")

# Stop if matching is incomplete or ambiguous; do not silently exclude anyone.
assert not participants["IDcode"].duplicated().any(), "Duplicate participant IDs in the symptom data."
assert set(participants["IDcode"]) == set(keep_ids), "Selected participants are missing from the symptom data."
assert np.isfinite(participants["Compulsivity"]).all(), "Some compulsivity scores are missing or invalid."
assert set(keep_ids).issubset(set(task["IDcode"])), "Selected participants are missing from the task data."

print(f"Verified: {len(participants)} participants with symptom scores and task data.")


Verified: 344 participants with symptom scores and task data.


**Five groups from low to high**

pd.qcut uses the 20th, 40th, 60th and 80th percentiles as boundaries. Each group therefore contains approximately 20% of participants. These are not five intervals of equal score width. All scores in the current selection are unique.

Tied scores are not arbitrarily split by participant ID. In future datasets, ties may produce unequal group sizes; if five distinct quantile intervals cannot be formed, qcut raises an error.

In [4]:
participants = participants.sort_values(["Compulsivity", "IDcode"]).reset_index(drop=True)
participants["compulsivity_group"], boundaries = pd.qcut(
    participants["Compulsivity"], q=5, labels=[1, 2, 3, 4, 5], retbins=True
)
participants["compulsivity_group"] = participants["compulsivity_group"].astype(int)

summary = participants.groupby("compulsivity_group").agg(
    n=("IDcode", "size"),
    min_compulsivity=("Compulsivity", "min"),
    max_compulsivity=("Compulsivity", "max"),
    mean_compulsivity=("Compulsivity", "mean"),
).reset_index()

assert participants["IDcode"].is_unique
assert set(participants["compulsivity_group"]) == {1, 2, 3, 4, 5}
assert summary["n"].sum() == len(keep_ids)
assert np.all(summary["max_compulsivity"].to_numpy()[:-1]
              <= summary["min_compulsivity"].to_numpy()[1:])

display(summary.round(4))
print("Quantile boundaries:", np.round(boundaries, 4))
display(participants.head())


,compulsivity_group,n,min_compulsivity,max_compulsivity,mean_compulsivity
0,1,69,-1.6624,-0.7454,-1.0308
1,2,69,-0.7222,-0.2830,-0.5171
2,3,68,-0.2790,0.1897,-0.0331
3,4,69,0.1919,0.8285,0.4844
4,5,69,0.8373,3.2687,1.5042


Quantile boundaries: [-1.6624 -0.7315 -0.2822  0.1915  0.832   3.2687]


,IDcode,Compulsivity,compulsivity_group
0,1112249,-1.662421,1
1,975696,-1.603693,1
2,961864,-1.530353,1
3,1074460,-1.418989,1
4,1029858,-1.402010,1


**Save group assignments**

We save one participant file containing all group labels, a summary, the exact quantile boundaries and five files listing the participants in each group. These five files contain participant IDs and scores; they can later be used to select each group’s complete task data. Rerunning the notebook overwrites these output files.

In [5]:
output_dir.mkdir(parents=True, exist_ok=True)
participants.to_csv(output_dir / "study1_compulsivity_groups.csv", index=False)
summary.to_csv(output_dir / "group_summary.csv", index=False)
pd.DataFrame({
    "quantile": [0, 0.2, 0.4, 0.6, 0.8, 1.0],
    "compulsivity_boundary": boundaries,
}).to_csv(output_dir / "quantile_boundaries.csv", index=False)

for group in range(1, 6):
    members = participants.loc[participants["compulsivity_group"] == group]
    members.to_csv(output_dir / f"group_{group}_participants.csv", index=False)
    print(f"Group {group}: {len(members)} participants")

print(f"Saved to: {output_dir.resolve()}")


Group 1: 69 participants
Group 2: 69 participants
Group 3: 68 participants
Group 4: 69 participants
Group 5: 69 participants
Saved to: /Users/daviddejonghe/Documents/Psychologie/Masterproef/simulationsHmfc/demo/Study1_Benwell/split_groups_compulsivity/study1_compulsivity_groepen
